<a href="https://colab.research.google.com/github/moshi516/summercamppruebapython/blob/main/1_Proyecto_finalipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Importamos las herramientas que usaremos hoy
import pandas as pd
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Creamos una conexión a un archivo de base de datos virtual en Colab
conexion = sqlite3.connect("Hidrocohete.db")

# 2. Leemos nuestro archivo CSV usando Pandas
df_original = pd.read_csv("hidrocohete.csv", encoding="latin1")

# 3. ¡La Magia! Guardamos toda esa información dentro de una tabla SQL llamada 'combates'
df_original.to_sql("Hidrocohete", conexion, if_exists="replace", index=False)

print("✅ ¡Éxito total! La base de datos relacional 'Hidrocohete.db' ha sido creada y la tabla 'hidrocohete' está lista.")

✅ ¡Éxito total! La base de datos relacional 'Hidrocohete.db' ha sido creada y la tabla 'hidrocohete' está lista.


In [ ]:
# 1. Instalar librerías de Python
!pip install streamlit plotly pandas -q

# 2. Descargar y preparar Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

### Generando y Ejecutando la Aplicación Streamlit

A continuación, crearemos un archivo `mi_app.py` que contendrá el código de tu aplicación Streamlit. Este archivo cargará los datos, creará el gráfico de barras y lo mostrará.


In [ ]:
%%writefile mi_app.py

import streamlit as st
import pandas as pd
import plotly.express as px

# Título principal de la aplicación Streamlit
st.title("Análisis de Lanzamientos de Hidrocohetes")

# Cargar los datos
# Asumimos que 'hidrocohete.csv' está disponible en el entorno de Colab
df_cohetes = pd.read_csv("hidrocohete.csv", encoding="latin1")

# --- GRÁFICO 1: BARRAS (Distancia Alcanzada por Participante) ---
st.subheader("1. Gráfico de Barras: Distancia Alcanzada por Participante")

fig_barras = px.bar(
    df_cohetes,
    x="Nombre",
    y="Distancia",
    # Se eliminó 'color="Presion_PSI"' ya que no es una columna válida en el CSV cargado.
    title="Distancia Alcanzada por Participante",
    template="plotly_dark"
)
st.plotly_chart(fig_barras, use_container_width=True)


# --- GRÁFICO 2: SCATTER PLOT (Distancia vs Cantidad de Agua, coloreado por Nombre) ---
st.subheader("2. Gráfico de Dispersión: Distancia vs. Cantidad de Agua")
fig_scatter_agua = px.scatter(
    df_cohetes,
    x="Cant_de_agua",
    y="Distancia",
    color="Nombre", # Diferenciar puntos por participante
    title="Relación entre Cantidad de Agua y Distancia Alcanzada por Participante",
    labels={"Cant_de_agua": "Cantidad de Agua (ml)", "Distancia": "Distancia (metros)"},
    template="plotly_dark"
)
st.plotly_chart(fig_scatter_agua, use_container_width=True)

# --- GRÁFICO 3: BOX PLOT (Distribución de Distancia por Sexo) ---
st.subheader("3. Gráfico de Caja: Distribución de Distancia por Sexo")
fig_box_sexo = px.box(
    df_cohetes,
    x="Sexo",
    y="Distancia",
    color="Sexo", # Colorear las cajas por sexo
    title="Distribución de la Distancia Alcanzada por Sexo",
    labels={"Sexo": "Género", "Distancia": "Distancia (metros)"},
    template="plotly_dark"
)
st.plotly_chart(fig_box_sexo, use_container_width=True)

# --- GRÁFICO 4: BARRAS (Cantidad Media de Agua por Edad) ---
st.subheader("4. Gráfico de Barras: Cantidad Media de Agua por Edad")

# Agrupar por edad y calcular la media de Cant_de_agua
df_agua_por_edad = df_cohetes.groupby('Edad')['Cant_de_agua'].mean().reset_index()

fig_agua_edad_barras = px.bar(
    df_agua_por_edad,
    x="Edad",
    y="Cant_de_agua",
    color="Edad", # Colorear por edad
    title="Cantidad Media de Agua Utilizada por Edad",
    labels={"Cant_de_agua": "Cantidad Media de Agua (ml)", "Edad": "Edad del Participante"},
    template="plotly_dark"
)
st.plotly_chart(fig_agua_edad_barras, use_container_width=True)

Overwriting mi_app.py


In [ ]:
# 3. Ejecutar la aplicación Streamlit en segundo plano
!streamlit run mi_app.py --server.port 8501 &>/dev/null&

# 4. Iniciar Cloudflare Tunnel para exponer la aplicación
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501 --no-autoupdate > portal.log 2>&1 &

# 5. Esperar un momento y mostrar la URL pública
import time
time.sleep(4) # Dale tiempo a cloudflared para arrancar
import re

with open("portal.log") as logfile:
    logurl = logfile.read()

m = re.search("https://.*[.]trycloudflare.com", logurl)

if m:
    print(f"Su aplicación Streamlit está disponible en: {m.group(0)}")
else:
    print("No se pudo obtener la URL de Cloudflare Tunnel. Verifique el archivo portal.log para más detalles.")


Su aplicación Streamlit está disponible en: https://fixes-opens-barriers-participate.trycloudflare.com
